In [43]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/val.txt
/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/test.txt
/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/train.txt


In [44]:
import numpy as np
import pandas as pd 
import matplotlib as plt
import seaborn as sns 

# 1. Text Preprocessing

In [45]:
df= pd.read_csv('/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/train.txt', sep= ';', header= None, names= ['text', 'emotion'])

In [46]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [47]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [48]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

## a. Emotion Encoding

In [49]:
unique_emotions= df['emotion'].unique()
emotion_numbers= {}
i=0
for emo in unique_emotions:
    emotion_numbers[emo]=i
    i+=1

df['emotion']= df['emotion'].map(emotion_numbers)


In [50]:
emotion_numbers

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [51]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


## b. Lowercasing

In [52]:
df['text'] = df['text'].apply(lambda x : x.lower())

## c. Removing punctuation

In [53]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [54]:
df['text'] = df['text'].apply(remove_punc)

## d. Removing numbers

In [55]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

## e. Removing emojis

In [56]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

## f. Removing Stopwords

In [57]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [58]:
print(stop_words)

{"we've", 't', 'has', "they'll", 'those', 'i', 'theirs', "doesn't", 'below', "isn't", 'y', 'when', 'other', 'his', "he's", 'between', 'and', 'until', 'nor', 'because', "it'd", 'shan', 'than', "they're", "weren't", "you've", 'once', "mightn't", 'on', 'any', 'can', 'during', 'into', 'of', 'by', 'had', 'did', 'it', 'at', "i've", 'she', 'these', "they'd", 'm', 'out', "hadn't", 'which', 'up', "i'd", 'about', 'while', 'been', 'should', 'mightn', 'who', 'now', 'is', 'all', 'then', 'o', 'be', 'to', 'more', 'ourselves', 'was', "shouldn't", 'own', "we're", 'am', 'only', 'against', "couldn't", 'very', "you'll", 'same', "we'll", 'didn', 'weren', 'haven', 've', 'does', 'again', 'isn', 'itself', 'needn', "won't", 'won', 'having', 'him', 'whom', 'yourselves', 'the', 'you', 'what', 'through', 'hadn', 'he', 'myself', 'wasn', 'this', "she's", 'have', 'if', 'for', 'here', "mustn't", 'before', 'doing', "wouldn't", 'but', "hasn't", "we'd", 'off', 'ain', 'over', "shan't", "they've", 'hasn', 'under', 'both',

In [59]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

## g. Tokenization

In [60]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [61]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [62]:
stop_words = set(stopwords.words('english'))

In [63]:
# Creating the text-cleaning function
def remove(txt):
    words = word_tokenize(txt)
    words = [word for word in words if word.lower() not in stop_words]
    return " ".join(words)

In [64]:
# Test
remove(df.loc[1, 'text'])

'go feeling hopeless damned hopeful around someone cares awake'

In [65]:
df['text'] = df['text'].apply(remove)

In [66]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


# 2. Model Training

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [68]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

## a. Bag-of-Words (BoW): Naive bayes

In [69]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [70]:
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.7678125


In [71]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

## b. TF-IDF (Term Frequency-Inverse Document Frequency): Naive bayes

In [72]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [73]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [74]:
y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


## c. Logistic Regression

In [75]:
from sklearn.linear_model import LogisticRegression

In [76]:
logistic_model = LogisticRegression(max_iter=1000)

In [77]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [78]:
log_pred = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test,log_pred ))

0.8615625


# Streamlit

In [81]:
import joblib
from sklearn.metrics import accuracy_score

# Calculate Logistic Regression accuracy
log_pred = logistic_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, log_pred)

# Save everything needed for Streamlit
model_data = {
    "vectorizer": tfidf_vectorizer,
    "model": logistic_model,
    "stop_words": stop_words,
    "emotion_numbers": emotion_numbers,
    "emotion_names": unique_emotions,
    "accuracy": accuracy
}

# Save model
joblib.dump(model_data, "emotion_model.pkl")

print("Model saved successfully!")
print(f"Logistic Regression Accuracy: {accuracy:.2%}")

Model saved successfully!
Logistic Regression Accuracy: 86.16%


In [82]:
# Load the saved model
model_data = joblib.load("emotion_model.pkl")

print(model_data.keys())

vectorizer = model_data["vectorizer"]
model = model_data["model"]

print(type(vectorizer))
print(type(model))
print(f"Accuracy: {model_data['accuracy']:.2%}")

dict_keys(['vectorizer', 'model', 'stop_words', 'emotion_numbers', 'emotion_names', 'accuracy'])
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'sklearn.linear_model._logistic.LogisticRegression'>
Accuracy: 86.16%
